# Mutual Information Evaluation

This notebook estimates the mutual information between the original and
anonymized embeddings using the MINDE estimator.

> **Important**
>
> This notebook requires the original MINDE repository and its dedicated
> Python environment.
>
> Repository:
> https://github.com/MustaphaBounoua/minde

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
MINDE_ROOT = PROJECT_ROOT / "minde"

if not MINDE_ROOT.exists():
    raise FileNotFoundError(
        "MINDE repository not found.\n"
        "Clone it into the project root as:\n"
        "project/minde/"
    )

sys.path.insert(0, str(MINDE_ROOT))

In [2]:
import torch
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

from src.libs.minde import MINDE
from src.scripts.helper import get_default_config

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device: cuda
NVIDIA GeForce GTX 1650


In [4]:
DATA_DIR = PROJECT_ROOT / "data"

INPUT_PATH = DATA_DIR / "case_embeddings.parquet"

ANON_DIR = DATA_DIR / "anonymized_outputs"

OUTPUT_DIR = DATA_DIR / "results"
OUTPUT_DIR.mkdir(exist_ok=True)

CLUSTER_K = [5, 10, 25, 50, 100]

In [5]:
X = pd.read_parquet(
    INPUT_PATH
).to_numpy(dtype=np.float32)
X = X[:1000]
print(X.shape)

(1000, 768)


In [6]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

In [7]:
class MINDataset(Dataset):

    def __init__(self, X, Y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32,
        )

        self.Y = torch.tensor(
            Y,
            dtype=torch.float32,
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return {
            "x": self.X[idx],
            "y": self.Y[idx],
        }

In [8]:
args = get_default_config()

args.type = "c"
args.importance_sampling = True
args.arch = "mlp"
args.use_ema = True

args.max_epochs = 150
args.warmup_epochs = 5
args.test_epoch = 5

args.lr = 1e-3
args.bs = 512

args.accelerator = (
    "gpu"
    if torch.cuda.is_available()
    else "cpu"
)

args.devices = 1

args.out_dir = "./outputs"
args.benchmark = "embedding_mi"
args.seed = 42

args.mc_iter = 20

In [9]:
def estimate_mi(
    original_embeddings,
    anonymized_embeddings,
):

    anonymized_embeddings = scaler.transform(
        anonymized_embeddings
    )

    dataset = MINDataset(
        original_embeddings,
        anonymized_embeddings,
    )

    loader = DataLoader(
        dataset,
        batch_size=args.bs,
        shuffle=True,
        num_workers=0,
    )

    model = MINDE(
        args,
        var_list={
            "x": original_embeddings.shape[1],
            "y": anonymized_embeddings.shape[1],
        },
        gt=None,
    )

    model.fit(
        loader,
        loader,
    )

    return model.compute_mi()

In [10]:
results = []

algorithms = {
    "MDAV-C": "mdav_c_k{}.parquet",
    "RMDAV-M": "rmdav_m_k{}.parquet",
}

for algorithm, pattern in algorithms.items():

    for cluster_k in CLUSTER_K:

        print(
            f"{algorithm} | k={cluster_k}"
        )

        X_anon = pd.read_parquet(
            ANON_DIR / pattern.format(cluster_k)
        ).to_numpy(dtype=np.float32)

        mi, mi_sigma = estimate_mi(
            X,
            X_anon,
        )

        results.append(
            {
                "algorithm": algorithm,
                "cluster_k": cluster_k,
                "mi": mi,
                "mi_sigma": mi_sigma,
            }
        )

MDAV-C | k=5


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: ./outputs/minde_c/embedding_mi/seed_42/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type           | Params
---------------------------------------------
0 | score     | UnetMLP_simple | 1.4 M 
1 | model_ema | EMA            | 1.4 M 
---------------------------------------------
2.9 M     Trainable params
0         Non-trainable params
2.9 M     Total params
11.596    Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

d:\Post UG\M.Tech\FYP\Syntactic Vector Anonymization\mi\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:492: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
d:\Post UG\M.Tech\FYP\Syntactic Vector Anonymization\mi\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
d:\Post UG\M.Tech\FYP\Syntactic Vector Anonymization\mi\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
d:\Post UG\M.Tech\FYP\Syntactic Vector Anonymization\mi\Lib\

Training: |          | 0/? [00:00<?, ?it/s]

...

Epoch:  50  GT:  Not given MINDE_estimate:  2.204 MINDE_sigma_estimate:  7.755
Epoch:  55  GT:  Not given MINDE_estimate:  2.869 MINDE_sigma_estimate:  8.844
Epoch:  60  GT:  Not given MINDE_estimate:  3.651 MINDE_sigma_estimate:  9.835
Epoch:  65  GT:  Not given MINDE_estimate:  4.547 MINDE_sigma_estimate:  11.262
Epoch:  70  GT:  Not given MINDE_estimate:  5.549 MINDE_sigma_estimate:  12.377
Epoch:  75  GT:  Not given MINDE_estimate:  6.697 MINDE_sigma_estimate:  13.502
Epoch:  80  GT:  Not given MINDE_estimate:  7.954 MINDE_sigma_estimate:  14.864
Epoch:  85  GT:  Not given MINDE_estimate:  9.381 MINDE_sigma_estimate:  16.24
Epoch:  90  GT:  Not given MINDE_estimate:  10.91 MINDE_sigma_estimate:  17.688
Epoch:  95  GT:  Not given MINDE_estimate:  12.517 MINDE_sigma_estimate:  19.033


Validation: |          | 0/? [00:00<?, ?it/s]

Epoch:  100  GT:  Not given MINDE_estimate:  14.28 MINDE_sigma_estimate:  20.387
Epoch:  105  GT:  Not given MINDE_estimate:  16.174 MINDE_sigma_estimate:  21.935
Epoch:  110  GT:  Not given MINDE_estimate:  18.122 MINDE_sigma_estimate:  23.402
Epoch:  115  GT:  Not given MINDE_estimate:  20.184 MINDE_sigma_estimate:  25.241
Epoch:  120  GT:  Not given MINDE_estimate:  22.28 MINDE_sigma_estimate:  26.36
Epoch:  125  GT:  Not given MINDE_estimate:  24.446 MINDE_sigma_estimate:  27.579
Epoch:  130  GT:  Not given MINDE_estimate:  26.622 MINDE_sigma_estimate:  29.259
Epoch:  135  GT:  Not given MINDE_estimate:  28.848 MINDE_sigma_estimate:  30.293
Epoch:  140  GT:  Not given MINDE_estimate:  31.139 MINDE_sigma_estimate:  32.276
Epoch:  145  GT:  Not given MINDE_estimate:  33.418 MINDE_sigma_estimate:  33.582


Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=150` reached.


In [11]:
results_df = pd.DataFrame(results)

results_df

,algorithm,cluster_k,mi,mi_sigma
0,MDAV-C,5,27.385625,27.609277
1,MDAV-C,10,20.268695,20.813159
2,MDAV-C,25,12.733525,12.846216
3,MDAV-C,50,7.975684,8.703833
4,MDAV-C,100,6.021649,6.703552
5,RMDAV-M,5,46.216722,51.617932
6,RMDAV-M,10,45.119236,49.536194
7,RMDAV-M,25,42.312817,43.152979
8,RMDAV-M,50,40.153137,41.886023
9,RMDAV-M,100,35.244640,34.830481


In [12]:
results_df.to_csv(
    OUTPUT_DIR / "mutual_information.csv",
    index=False,
)

print("Results saved successfully.")

Results saved successfully.
